In [17]:
#ABCD
import json
import random
import os
from tqdm import tqdm
from datasets import load_dataset, Dataset

# Create output directory
os.makedirs("./MMLU_PRO", exist_ok=True)

# Load the diamond split
test_set = load_dataset("rbiswasfc/MMLU-Pro", split="test")

# Processed and formatted dataset
filtered_mmlu_pro_test = []

for idx, item in enumerate(tqdm(test_set), 0):
    # Clean whitespace from answers
    correct_answer = item['options'][item['answer_index']].strip()
    incorrect_answers = [
        item['options'][key].strip()
        for key in range(len(item['options']))
        if key != item['answer_index']
    ]

    # Create and shuffle answer pool
    answers = [('Correct Answer', correct_answer)]
    answers += [(f'Incorrect Answer {i+1}', ans) for i, ans in enumerate(incorrect_answers)]
    random.seed(42)
    random.shuffle(answers)

    # Assign A–D (or more letters if needed) and format choices
    choice_labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
    choice_dict = {}
    formatted_choices = []
    correct_choice = None

    for i, (label, answer) in enumerate(answers):
        choice_letter = choice_labels[i]
        choice_dict[choice_letter] = answer
        formatted_choices.append(f"({choice_letter}) {answer}")
        if label == 'Correct Answer':
            correct_choice = choice_letter

    # Format the full question
    formatted_question = f"{item['question'].strip()} Choices:\n" + "\n".join(formatted_choices) + "\n"

    # Add to final dataset
    filtered_mmlu_pro_test.append({
        'id': idx,
        'category': item['category'],
        'Question': formatted_question,
        'Choices': choice_dict,
        'Correct Choice': correct_choice
    })

# Save to JSON
with open("./MMLU_PRO/test.json", "w", encoding="utf-8") as f:
    json.dump(filtered_mmlu_pro_test, f, ensure_ascii=False, indent=4)

# Push to Hugging Face Hub
dataset = Dataset.from_list(filtered_mmlu_pro_test)
dataset.push_to_hub('talzoomanzoo/mmlu_pro_raw', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.16s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/mmlu_pro_raw/commit/994dd5710f0039f6c9a8e116bdb002f3dc257a23', commit_message='Upload dataset', commit_description='', oid='994dd5710f0039f6c9a8e116bdb002f3dc257a23', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/mmlu_pro_raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/mmlu_pro_raw'), pr_revision=None, pr_num=None)

In [ ]:
"""Data preprocess for GPQA
Data Link: https://huggingface.co/datasets/Idavidrein/gpqa
"""
import csv
import json
import random
from tqdm import tqdm

# Paths to data
split = 'diamond'  # diamond, main, extended
data_path = f'./GPQA/original_data/gpqa_{split}.csv'
output_path = f'./GPQA/{split}.json'

# Define the keys we want to keep
keys_to_keep = [
    'id',
    'Question',
    'Subdomain',
    'High-level domain',
    'Correct Answer',
    'Incorrect Answer 1',
    'Incorrect Answer 2',
    'Incorrect Answer 3'
]

filtered_data = []
with open(data_path, mode='r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for idx, row in enumerate(tqdm(csv_reader), 0):
        # Add id field
        row['id'] = idx
        # Create new dictionary with only desired keys
        filtered_row = {key: row[key] for key in keys_to_keep}

        # Extract answers and shuffle them
        answers = [
            ('Correct Answer', filtered_row['Correct Answer']),
            ('Incorrect Answer 1', filtered_row['Incorrect Answer 1']),
            ('Incorrect Answer 2', filtered_row['Incorrect Answer 2']),
            ('Incorrect Answer 3', filtered_row['Incorrect Answer 3'])
        ]
        random.shuffle(answers)

        # Assign new choices A, B, C, D in order and determine the correct choice
        choices = ['A', 'B', 'C', 'D']
        formatted_answers = []
        correct_choice = None
        for i, (label, answer) in enumerate(answers):
            choice = choices[i]
            formatted_answers.append((choice, answer))
            if label == 'Correct Answer':
                correct_choice = choice

        # Update the Question field
        formatted_choices = "\n".join([f"({choice}) {answer}" for choice, answer in formatted_answers])
        filtered_row['Question'] = f"{filtered_row['Question']} Choices:\n{formatted_choices}\n"

        # Add the Correct Choice field
        filtered_row['Correct Choice'] = correct_choice

        # Append the updated row to filtered_data
        filtered_data.append(filtered_row)

# Write the updated data to JSON
with open(output_path, mode='w', encoding='utf-8') as json_file:
    json.dump(filtered_data, json_file, indent=4, ensure_ascii=False)

In [1]:
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset
import os

if not os.path.exists("./BRUMO"):
    os.makedirs("./BRUMO")

train_set = load_dataset("MathArena/brumo_2025", split="train")
with open("./BRUMO/test.json", "w") as f:
    aime_train = []
    for item in train_set:
        aime_train.append({
            'problem_idx': item['problem_idx'],
            'Question': item['problem'],
            'answer': item['answer'],
            'problem_type': item['problem_type'],
        })
    json.dump(aime_train, f, ensure_ascii=False, indent=4)
dataset = Dataset.from_list(aime_train)
dataset.push_to_hub('talzoomanzoo/brumo2025', private=False)

/home/mjgwak/miniconda3/envs/uid_2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.90s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/brumo2025/commit/69ea201ce4a497bb3691701515aec8c652b118dc', commit_message='Upload dataset', commit_description='', oid='69ea201ce4a497bb3691701515aec8c652b118dc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/brumo2025', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/brumo2025'), pr_revision=None, pr_num=None)

In [2]:
import pandas as pd
from datasets import load_dataset
import os

#wrong format

def download_superGPQA_to_csv(output_dir="./SUPERGPQA/original_data", filename="SUPERGPQA.csv"):
    """
    Download the SUPERGPQA dataset from Hugging Face and save to CSV file.
    
    Args:
        output_dir (str): Directory to save the CSV file
        filename (str): Name of the CSV file
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Load the dataset from Hugging Face
        print("Loading SUPERGPQA dataset from Hugging Face...")
        dataset = load_dataset("m-a-p/SuperGPQA", split="train")
        
        print(f"Dataset loaded successfully! Found {len(dataset)} examples")
        print(f"Dataset features: {list(dataset.features.keys())}")
        
        # Convert to pandas DataFrame
        print("Converting to DataFrame...")
        df = dataset.to_pandas()
        
        # Save to CSV
        output_path = os.path.join(output_dir, filename)
        print(f"Saving to {output_path}...")
        df.to_csv(output_path, index=False, encoding='utf-8')
        
        # Display some basic info about the dataset
        print("\nDataset Info:")
        print(f"- Total rows: {len(df)}")
        print(f"- Columns: {list(df.columns)}")
        print(f"- File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
        
        return df
        
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        return None
    
    
if __name__ == "__main__":
    download_superGPQA_to_csv()


Loading SUPERGPQA dataset from Hugging Face...
Dataset loaded successfully! Found 26529 examples
Dataset features: ['uuid', 'question', 'options', 'answer', 'answer_letter', 'discipline', 'field', 'subfield', 'difficulty', 'is_calculation']
Converting to DataFrame...
Saving to ./SUPERGPQA/original_data/SUPERGPQA.csv...

Dataset Info:
- Total rows: 26529
- Columns: ['uuid', 'question', 'options', 'answer', 'answer_letter', 'discipline', 'field', 'subfield', 'difficulty', 'is_calculation']
- File size: 18.79 MB


In [6]:
"""Data preprocess for SUPERGPQA
Data Link: https://huggingface.co/datasets/Idavidrein/gpqa
"""
import csv
import json
import random
from tqdm import tqdm

# Paths to data
split = 'diamond'  # diamond, main, extended
data_path = f'./SUPERGPQA/original_data/SUPERGPQA.csv'
output_path = f'./SUPERGPQA/train.json'

# Define the keys we want to keep
keys_to_keep = [
    'uuid',
    'question',
    'options',
    'discipline',
    'field',
    'subfield',
    'difficulty',
    'is_calculation'
]

filtered_data = []
with open(data_path, mode='r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for idx, row in enumerate(tqdm(csv_reader), 0):
        # Add id field
        row['id'] = idx
        # Create new dictionary with only desired keys
        filtered_row = {key: row[key] for key in keys_to_keep}

        # Extract answers and shuffle them
        answers = filtered_row['options']

        # Assign new choices A, B, C, D in order and determine the correct choice
        choices = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
        formatted_answers = []
        correct_choice = None
        for i, (label, answer) in enumerate(answers):
            choice = choices[i]
            formatted_answers.append((choice, answer))
            if label == 'Correct Answer':
                correct_choice = choice

        # Update the Question field
        formatted_choices = "\n".join([f"({choice}) {answer}" for choice, answer in formatted_answers])
        filtered_row['Question'] = f"{filtered_row['Question']} Choices:\n{formatted_choices}\n"

        # Add the Correct Choice field
        filtered_row['Correct Choice'] = correct_choice

        # Append the updated row to filtered_data
        filtered_data.append(filtered_row)

# Write the updated data to JSON
with open(output_path, mode='w', encoding='utf-8') as json_file:
    json.dump(filtered_data, json_file, indent=4, ensure_ascii=False)

0it [00:00, ?it/s]


ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset
import os

if not os.path.exists("./OLYMPIADBENCH"):
    os.makedirs("./OLYMPIADBENCH")

test_set1 = load_dataset("Hothan/OlympiadBench", 'OE_TO_maths_en_COMP', split="train") #674
test_set2 = load_dataset("Hothan/OlympiadBench", 'OE_TO_maths_en_COMP', split="train")

with open("./OLYMPIADBENCH/test.json", "w") as f:
    aime_test = []
    for item in test_set1:
        aime_test.append({
            'Question': item['question'],
            'answer': item['final_answer'],
        })
    for item in test_set2:
        aime_test.append({
            'Question': item['question'],
            'answer': item['final_answer'],
        })

    print(len(aime_test))
    json.dump(aime_test, f, ensure_ascii=False, indent=4)


dataset = Dataset.from_list(aime_test)
dataset.push_to_hub('talzoomanzoo/olympiadbench', private=False)

1348


Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.48 shards/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/olympiadbench/commit/58a71ac74e0cc78463bddb343bd98e0201ce90f1', commit_message='Upload dataset', commit_description='', oid='58a71ac74e0cc78463bddb343bd98e0201ce90f1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/olympiadbench', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/olympiadbench'), pr_revision=None, pr_num=None)

In [ ]:
#gsm8k
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset, DatasetDict
import os

#hmmt2025

if not os.path.exists("./HMMT"): 
    os.makedirs("./HMMT")

test_set = load_dataset("MathArena/hmmt_feb_2025", split="train")


# Prepare test data
hmmt_test = [] #30
for item in test_set:
    hmmt_test.append({
        'Question': item['problem'],
        'answer': item['answer'],
    })

# Save individual files
with open("./HMMT/test.json", "w") as f:
    json.dump(hmmt_test, f, ensure_ascii=False, indent=4)

# Create DatasetDict with both splits
dataset_dict = DatasetDict({
    'test': Dataset.from_list(hmmt_test)
})

# Upload to hub with both splits
dataset_dict.push_to_hub('talzoomanzoo/hmmt', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.25s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/hmmt/commit/8602b342733ff6e33ab848936752a905478a5745', commit_message='Upload dataset', commit_description='', oid='8602b342733ff6e33ab848936752a905478a5745', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/hmmt', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/hmmt'), pr_revision=None, pr_num=None)

In [ ]:
#gsm8k
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset, DatasetDict
import os

if not os.path.exists("./MINERVA_MATH"):
    os.makedirs("./MINERVA_MATH")

test_set = load_dataset("math-ai/minervamath", split="test")


# Prepare test data
minerva_math_test = [] #272
for item in test_set:
    minerva_math_test.append({
        'Question': item['question'],
        'answer': item['answer'],
    })

# Save individual files
with open("./MINERVA_MATH/test.json", "w") as f:
    json.dump(minerva_math_test, f, ensure_ascii=False, indent=4)

# Create DatasetDict with both splits
dataset_dict = DatasetDict({
    'test': Dataset.from_list(minerva_math_test)
})

# Upload to hub with both splits
dataset_dict.push_to_hub('talzoomanzoo/minervamath', private=False)

/home/mjgwak/miniconda3/envs/uid_2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.27s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/minervamath/commit/fc64b615c00e77b6c6bf9077815508779b5b7c04', commit_message='Upload dataset', commit_description='', oid='fc64b615c00e77b6c6bf9077815508779b5b7c04', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/minervamath', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/minervamath'), pr_revision=None, pr_num=None)

In [ ]:
import pandas as pd
from datasets import load_dataset
import os

#wrong format

def download_gpqa_diamond_to_csv(output_dir="./GPQA/original_data", filename="gpqa_diamond.csv"):
    """
    Download the GPQA dataset from Hugging Face and save to CSV file.
    
    Args:
        output_dir (str): Directory to save the CSV file
        filename (str): Name of the CSV file
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Load the dataset from Hugging Face
        print("Loading GPQA dataset from Hugging Face...")
        dataset = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
        
        print(f"Dataset loaded successfully! Found {len(dataset)} examples")
        print(f"Dataset features: {list(dataset.features.keys())}")
        
        # Convert to pandas DataFrame
        print("Converting to DataFrame...")
        df = dataset.to_pandas()
        
        # Save to CSV
        output_path = os.path.join(output_dir, filename)
        print(f"Saving to {output_path}...")
        df.to_csv(output_path, index=False, encoding='utf-8')
        
        # Display some basic info about the dataset
        print("\nDataset Info:")
        print(f"- Total rows: {len(df)}")
        print(f"- Columns: {list(df.columns)}")
        print(f"- File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
        
        return df
        
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        return None
    

def download_gpqa_main_to_csv(output_dir="./GPQA/original_data", filename="gpqa_main.csv"):
    """
    Download the GPQA dataset from Hugging Face and save to CSV file.
    
    Args:
        output_dir (str): Directory to save the CSV file
        filename (str): Name of the CSV file
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Load the dataset from Hugging Face
        print("Loading GPQA dataset from Hugging Face...")
        dataset = load_dataset("Idavidrein/gpqa", "gpqa_main", split="train")
        
        print(f"Dataset loaded successfully! Found {len(dataset)} examples")
        print(f"Dataset features: {list(dataset.features.keys())}")
        
        # Convert to pandas DataFrame
        print("Converting to DataFrame...")
        df = dataset.to_pandas()
        
        # Save to CSV
        output_path = os.path.join(output_dir, filename)
        print(f"Saving to {output_path}...")
        df.to_csv(output_path, index=False, encoding='utf-8')
        
        # Display some basic info about the dataset
        print("\nDataset Info:")
        print(f"- Total rows: {len(df)}")
        print(f"- Columns: {list(df.columns)}")
        print(f"- File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
        
        return df
        
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        return None
    

def download_gpqa_extended_to_csv(output_dir="./GPQA/original_data", filename="gpqa_extended.csv"):
    """
    Download the GPQA dataset from Hugging Face and save to CSV file.
    
    Args:
        output_dir (str): Directory to save the CSV file
        filename (str): Name of the CSV file
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Load the dataset from Hugging Face
        print("Loading GPQA dataset from Hugging Face...")
        dataset = load_dataset("Idavidrein/gpqa", "gpqa_extended", split="train")
        
        print(f"Dataset loaded successfully! Found {len(dataset)} examples")
        print(f"Dataset features: {list(dataset.features.keys())}")
        
        # Convert to pandas DataFrame
        print("Converting to DataFrame...")
        df = dataset.to_pandas()
        
        # Save to CSV
        output_path = os.path.join(output_dir, filename)
        print(f"Saving to {output_path}...")
        df.to_csv(output_path, index=False, encoding='utf-8')
        
        # Display some basic info about the dataset
        print("\nDataset Info:")
        print(f"- Total rows: {len(df)}")
        print(f"- Columns: {list(df.columns)}")
        print(f"- File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
        
        return df
        
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        return None
    
if __name__ == "__main__":
    download_gpqa_diamond_to_csv()
    download_gpqa_main_to_csv()
    download_gpqa_extended_to_csv()


Loading GPQA dataset from Hugging Face...


Old caching folder /home/mjgwak/.cache/huggingface/datasets/Idavidrein___gpqa/gpqa_diamond/0.0.0/90b8e5be2b1d3d2dbfe016cdab47981150600c4a for dataset gpqa exists but no data were found. Removing it. 
Generating train split: 100%|██████████| 198/198 [00:00<00:00, 6674.59 examples/s]


Dataset loaded successfully! Found 198 examples
Dataset features: ['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3', 'Pre-Revision Explanation', 'Self-reported question-writing time (minutes)', 'Question', 'Correct Answer', 'Incorrect Answer 1', 'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation', 'Revision Comments (from Question Writer)', 'Subdomain', "Writer's Difficulty Estimate", 'Extra Revised Question', 'Extra Revised Explanation', 'Extra Revised Correct Answer', 'Extra Revised Incorrect Answer 1', 'Extra Revised Incorrect Answer 2', 'Extra Revised Incorrect Answer 3', 'Non-Expert Validator Accuracy', 'Majority Non-Expert Vals Incorrect', 'Expert Validator Accuracy', 'Record ID', 'High-level domain', 'Question Writer', 'Feedback_EV_1', 'Validator Revision Suggestion_EV_1', 'Is First Validation_EV_1', 'Post hoc agreement_EV_1', 'Sufficient Expertise?_EV_1', 'Unders

Generating train split: 100%|██████████| 448/448 [00:00<00:00, 10099.53 examples/s]


Dataset loaded successfully! Found 448 examples
Dataset features: ['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3', 'Pre-Revision Explanation', 'Self-reported question-writing time (minutes)', 'Question', 'Correct Answer', 'Incorrect Answer 1', 'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation', 'Revision Comments (from Question Writer)', 'Subdomain', "Writer's Difficulty Estimate", 'Extra Revised Question', 'Extra Revised Explanation', 'Extra Revised Correct Answer', 'Extra Revised Incorrect Answer 1', 'Extra Revised Incorrect Answer 2', 'Extra Revised Incorrect Answer 3', 'Non-Expert Validator Accuracy', 'Majority Non-Expert Vals Incorrect', 'Expert Validator Accuracy', 'Record ID', 'High-level domain', 'Question Writer', 'Feedback_EV_1', 'Validator Revision Suggestion_EV_1', 'Is First Validation_EV_1', 'Post hoc agreement_EV_1', 'Sufficient Expertise?_EV_1', 'Unders

Generating train split: 100%|██████████| 546/546 [00:00<00:00, 9448.07 examples/s]

Dataset loaded successfully! Found 546 examples
Dataset features: ['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3', 'Pre-Revision Explanation', 'Self-reported question-writing time (minutes)', 'Question', 'Correct Answer', 'Incorrect Answer 1', 'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation', 'Revision Comments (from Question Writer)', 'Subdomain', "Writer's Difficulty Estimate", 'Extra Revised Question', 'Extra Revised Explanation', 'Extra Revised Correct Answer', 'Extra Revised Incorrect Answer 1', 'Extra Revised Incorrect Answer 2', 'Extra Revised Incorrect Answer 3', 'Non-Expert Validator Accuracy', 'Majority Non-Expert Vals Incorrect', 'Expert Validator Accuracy', 'Record ID', 'High-level domain', 'Question Writer', 'Feedback_EV_1', 'Validator Revision Suggestion_EV_1', 'Is First Validation_EV_1', 'Post hoc agreement_EV_1', 'Sufficient Expertise?_EV_1', 'Unders

In [7]:
"""Data preprocess for GPQA
Data Link: https://huggingface.co/datasets/Idavidrein/gpqa
"""
import csv
import json
import random
from tqdm import tqdm

# Paths to data
split = 'diamond'  # diamond, main, extended
data_path = f'./GPQA/original_data/gpqa_{split}.csv'
output_path = f'./GPQA/{split}.json'

# Define the keys we want to keep
keys_to_keep = [
    'id',
    'Question',
    'Subdomain',
    'High-level domain',
    'Correct Answer',
    'Incorrect Answer 1',
    'Incorrect Answer 2',
    'Incorrect Answer 3'
]

filtered_data = []
with open(data_path, mode='r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for idx, row in enumerate(tqdm(csv_reader), 0):
        # Add id field
        row['id'] = idx
        # Create new dictionary with only desired keys
        filtered_row = {key: row[key] for key in keys_to_keep}

        # Extract answers and shuffle them
        answers = [
            ('Correct Answer', filtered_row['Correct Answer']),
            ('Incorrect Answer 1', filtered_row['Incorrect Answer 1']),
            ('Incorrect Answer 2', filtered_row['Incorrect Answer 2']),
            ('Incorrect Answer 3', filtered_row['Incorrect Answer 3'])
        ]
        random.shuffle(answers)

        # Assign new choices A, B, C, D in order and determine the correct choice
        choices = ['A', 'B', 'C', 'D']
        formatted_answers = []
        correct_choice = None
        for i, (label, answer) in enumerate(answers):
            choice = choices[i]
            formatted_answers.append((choice, answer))
            if label == 'Correct Answer':
                correct_choice = choice

        # Update the Question field
        formatted_choices = "\n".join([f"({choice}) {answer}" for choice, answer in formatted_answers])
        filtered_row['Question'] = f"{filtered_row['Question']} Choices:\n{formatted_choices}\n"

        # Add the Correct Choice field
        filtered_row['Correct Choice'] = correct_choice

        # Append the updated row to filtered_data
        filtered_data.append(filtered_row)

# Write the updated data to JSON
with open(output_path, mode='w', encoding='utf-8') as json_file:
    json.dump(filtered_data, json_file, indent=4, ensure_ascii=False)

198it [00:00, 17781.61it/s]


In [9]:
#ABCD
import json
import random
import os
from tqdm import tqdm
from datasets import load_dataset, Dataset

# Create output directory
os.makedirs("./GPQA_DIAMOND", exist_ok=True)

# Load the diamond split
test_set = load_dataset("Idavidrein/gpqa", 'gpqa_diamond', split="train")

# Processed and formatted dataset
filtered_gpqa_diamond_test = []

for idx, item in enumerate(tqdm(test_set), 0):
    # Clean whitespace from answersls
    correct_answer = item['Correct Answer'].strip()
    incorrect_answers = [
        item['Incorrect Answer 1'].strip(),
        item['Incorrect Answer 2'].strip(),
        item['Incorrect Answer 3'].strip()
    ]

    # Create and shuffle answer pool
    answers = {'Correct Answer': correct_answer}
    answers.update({f'Incorrect Answer {i+1}': ans for i, ans in enumerate(incorrect_answers)})
    answers = list(answers.values())

    # Format choices
    # choice_labels = ['A', 'B', 'C', 'D']
    choice_list = []
    correct_choice = None

    for i, answer in enumerate(answers):
        choice_list.append(answer)
        if answer == correct_answer:
            correct_choice = answer

    # Format the full question
    # formatted_question = f"{item['Question'].strip()} Choices:\n" + "\n".join(formatted_choices) + "\n"
    random.seed(42)
    choice_list = random.shuffle(choice_list)
    # Add to final dataset
    filtered_gpqa_diamond_test.append({
        'id': idx,
        'Subdomain': item['Subdomain'],
        'High-level domain': item['High-level domain'],
        'Question': item['Question'],
        'Choices': choice_list,
        'Correct Choice': item['Correct Answer']
    })

# Save to JSON
with open("./GPQA_DIAMOND/test.json", "w", encoding="utf-8") as f:
    json.dump(filtered_gpqa_diamond_test, f, ensure_ascii=False, indent=4)

# Push to Hugging Face Hub
dataset = Dataset.from_list(filtered_gpqa_diamond_test)
dataset.push_to_hub('talzoomanzoo/gpqa_diamond_raw', private=False)


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.67s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/gpqa_diamond_raw/commit/3ac0c05b7e67fad1df3e16bc3ab312c549999647', commit_message='Upload dataset', commit_description='', oid='3ac0c05b7e67fad1df3e16bc3ab312c549999647', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/gpqa_diamond_raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/gpqa_diamond_raw'), pr_revision=None, pr_num=None)

In [12]:
#ABCD
import json
import random
import os
from tqdm import tqdm
from datasets import load_dataset, Dataset

# Create output directory
os.makedirs("./GPQA_DIAMOND", exist_ok=True)

# Load the diamond split
test_set = load_dataset("Idavidrein/gpqa", 'gpqa_diamond', split="train")

# Processed and formatted dataset
filtered_gpqa_diamond_test = []

for idx, item in enumerate(tqdm(test_set), 0):
    # Clean whitespace from answers
    correct_answer = item['Correct Answer'].strip()
    incorrect_answers = [
        item['Incorrect Answer 1'].strip(),
        item['Incorrect Answer 2'].strip(),
        item['Incorrect Answer 3'].strip()
    ]

    # Create and shuffle answer pool
    answers = [('Correct Answer', correct_answer)]
    answers += [(f'Incorrect Answer {i+1}', ans) for i, ans in enumerate(incorrect_answers)]
    random.seed(42)
    random.shuffle(answers)

    # Assign A–D and format choices
    choice_labels = ['A', 'B', 'C', 'D']
    choice_dict = {}
    formatted_choices = []
    correct_choice = None

    for i, (label, answer) in enumerate(answers):
        choice_letter = choice_labels[i]
        choice_dict[choice_letter] = answer
        formatted_choices.append(f"({choice_letter}) {answer}")
        if label == 'Correct Answer':
            correct_choice = choice_letter

    # Format the full question
    formatted_question = f"{item['Question'].strip()} Choices:\n" + "\n".join(formatted_choices) + "\n"

    # Add to final dataset
    filtered_gpqa_diamond_test.append({
        'id': idx,
        'Subdomain': item['Subdomain'],
        'High-level domain': item['High-level domain'],
        'Question': formatted_question,
        'Choices': choice_dict,
        'Correct Choice': correct_choice
    })

# Save to JSON
with open("./GPQA_DIAMOND/test.json", "w", encoding="utf-8") as f:
    json.dump(filtered_gpqa_diamond_test, f, ensure_ascii=False, indent=4)

# Push to Hugging Face Hub
dataset = Dataset.from_list(filtered_gpqa_diamond_test)
dataset.push_to_hub('talzoomanzoo/gpqa_diamond', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.05 shards/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/gpqa_diamond/commit/ff55adbeecdcc6f8697c6a4a3212b96785ac47c9', commit_message='Upload dataset', commit_description='', oid='ff55adbeecdcc6f8697c6a4a3212b96785ac47c9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/gpqa_diamond', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/gpqa_diamond'), pr_revision=None, pr_num=None)

In [10]:
#ABCD
import json
import random
import os
from tqdm import tqdm
from datasets import load_dataset, Dataset

temp_cache_dir = "/home/mjgwak/.cache/tmp_hf_cache"
os.makedirs(temp_cache_dir, exist_ok=True)

# Create output directory
os.makedirs("./GPQA_EXTENDED", exist_ok=True)

# Load the diamond split
test_set = load_dataset("Idavidrein/gpqa", "gpqa_extended", split="train", cache_dir=temp_cache_dir)

# Processed and formatted dataset
filtered_gpqa_extended_test = []

for idx, item in enumerate(tqdm(test_set), 0):
    # Clean whitespace from answers
    correct_answer = item['Correct Answer'].strip()
    incorrect_answers = [
        item['Incorrect Answer 1'].strip(),
        item['Incorrect Answer 2'].strip(),
        item['Incorrect Answer 3'].strip()
    ]

    # Create and shuffle answer pool
    answers = [('Correct Answer', correct_answer)]
    answers += [(f'Incorrect Answer {i+1}', ans) for i, ans in enumerate(incorrect_answers)]
    random.seed(42)
    random.shuffle(answers)

    # Assign A–D and format choices
    choice_labels = ['A', 'B', 'C', 'D']
    choice_dict = {}
    formatted_choices = []
    correct_choice = None

    for i, (label, answer) in enumerate(answers):
        choice_letter = choice_labels[i]
        choice_dict[choice_letter] = answer
        formatted_choices.append(f"({choice_letter}) {answer}")
        if label == 'Correct Answer':
            correct_choice = choice_letter

    # Format the full question
    formatted_question = f"{item['Question'].strip()} Choices:\n" + "\n".join(formatted_choices) + "\n"

    # Add to final dataset
    filtered_gpqa_extended_test.append({
        'id': idx,
        'Subdomain': item['Subdomain'],
        'High-level domain': item['High-level domain'],
        'Question': formatted_question,
        'Choices': choice_dict,
        'Correct Choice': correct_choice
    })

# Save to JSON
with open("./GPQA_EXTENDED/test.json", "w", encoding="utf-8") as f:
    json.dump(filtered_gpqa_extended_test, f, ensure_ascii=False, indent=4)

# Push to Hugging Face Hub
dataset = Dataset.from_list(filtered_gpqa_extended_test)
dataset.push_to_hub('talzoomanzoo/gpqa_extended', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.26 shards/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/gpqa_extended/commit/b8d3e9017ed56da265461393399103b0ade18ffb', commit_message='Upload dataset', commit_description='', oid='b8d3e9017ed56da265461393399103b0ade18ffb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/gpqa_extended', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/gpqa_extended'), pr_revision=None, pr_num=None)

In [11]:
#ABCD
import json
import random
import os
from tqdm import tqdm
from datasets import load_dataset, Dataset

temp_cache_dir = "/home/mjgwak/.cache/tmp_hf_cache"
os.makedirs(temp_cache_dir, exist_ok=True)

# Create output directory
os.makedirs("./GPQA_MAIN", exist_ok=True)

# Load the diamond split
test_set = load_dataset("Idavidrein/gpqa", "gpqa_main", split="train", cache_dir=temp_cache_dir)

# Processed and formatted dataset
filtered_gpqa_main_test = []

for idx, item in enumerate(tqdm(test_set), 0):
    # Clean whitespace from answers
    correct_answer = item['Correct Answer'].strip()
    incorrect_answers = [
        item['Incorrect Answer 1'].strip(),
        item['Incorrect Answer 2'].strip(),
        item['Incorrect Answer 3'].strip()
    ]

    # Create and shuffle answer pool
    answers = [('Correct Answer', correct_answer)]
    answers += [(f'Incorrect Answer {i+1}', ans) for i, ans in enumerate(incorrect_answers)]
    random.seed(42)
    random.shuffle(answers)

    # Assign A–D and format choices
    choice_labels = ['A', 'B', 'C', 'D']
    choice_dict = {}
    formatted_choices = []
    correct_choice = None

    for i, (label, answer) in enumerate(answers):
        choice_letter = choice_labels[i]
        choice_dict[choice_letter] = answer
        formatted_choices.append(f"({choice_letter}) {answer}")
        if label == 'Correct Answer':
            correct_choice = choice_letter

    # Format the full question
    formatted_question = f"{item['Question'].strip()} Choices:\n" + "\n".join(formatted_choices) + "\n"

    # Add to final dataset
    filtered_gpqa_main_test.append({
        'id': idx,
        'Subdomain': item['Subdomain'],
        'High-level domain': item['High-level domain'],
        'Question': formatted_question,
        'Choices': choice_dict,
        'Correct Choice': correct_choice
    })

# Save to JSON
with open("./GPQA_MAIN/test.json", "w", encoding="utf-8") as f:
    json.dump(filtered_gpqa_main_test, f, ensure_ascii=False, indent=4)

# Push to Hugging Face Hub
dataset = Dataset.from_list(filtered_gpqa_main_test)
dataset.push_to_hub('talzoomanzoo/gpqa_main', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/ shards]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/gpqa_main/commit/514362349ce5a4768658a01d58479e953b89d2ec', commit_message='Upload dataset', commit_description='', oid='514362349ce5a4768658a01d58479e953b89d2ec', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/gpqa_main', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/gpqa_main'), pr_revision=None, pr_num=None)

In [ ]:
#gsm8k
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset, DatasetDict
import os

if not os.path.exists("./GSM8K"):
    os.makedirs("./GSM8K")

train_set = load_dataset("skrishna/gsm8k_only_answer", split="train")
test_set = load_dataset("skrishna/gsm8k_only_answer", split="test")

# Prepare train data
gsm8k_train = []
for item in train_set:
    gsm8k_train.append({
        'Question': item['text'],
        'answer': item['label'],
    })

# Prepare test data
gsm8k_test = []
for item in test_set: #1319
    gsm8k_test.append({
        'Question': item['text'],
        'answer': item['label'],
    })

# Save individual files
with open("./GSM8K/train.json", "w") as f:
    json.dump(gsm8k_train, f, ensure_ascii=False, indent=4)

with open("./GSM8K/test.json", "w") as f:
    json.dump(gsm8k_test, f, ensure_ascii=False, indent=4)

# Create DatasetDict with both splits
dataset_dict = DatasetDict({
    'train': Dataset.from_list(gsm8k_train),
    'test': Dataset.from_list(gsm8k_test)
})

# Upload to hub with both splits
dataset_dict.push_to_hub('talzoomanzoo/gsm8k', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.62s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/gsm8k/commit/46501e87b15df04e7d42dd934f065be523c8a13c', commit_message='Upload dataset', commit_description='', oid='46501e87b15df04e7d42dd934f065be523c8a13c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/gsm8k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/gsm8k'), pr_revision=None, pr_num=None)

In [2]:
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset
import os

if not os.path.exists("./AIME"):
    os.makedirs("./AIME")

train_set = load_dataset("gneubig/aime-1983-2024", split="train")
test_set1 = load_dataset("opencompass/AIME2025", 'AIME2025-I', split="test")
test_set2 = load_dataset("opencompass/AIME2025", 'AIME2025-II', split="test")
with open("./AIME/train.json", "w") as f:
    aime_train = []
    for item in train_set:
        aime_train.append({
            'id': item['ID'],
            'year': item['Year'],
            'problem_number': item['Problem Number'],
            'Question': item['Question'],
            'answer': item['Answer'],
        })
    json.dump(aime_train, f, ensure_ascii=False, indent=4)

with open("./AIME/test.json", "w") as f:
    aime_test = []
    for item in test_set1:
        aime_test.append({
            'Question': item['question'],
            'answer': item['answer'],
        })
    for item in test_set2:
        aime_test.append({
            'Question': item['question'],
            'answer': item['answer'],
        })
    json.dump(aime_test, f, ensure_ascii=False, indent=4)

dataset = Dataset.from_list(aime_train)
dataset.push_to_hub('talzoomanzoo/aime_train', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.75s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/aime_train/commit/b8c17c666e0c1d0ac1e99dc7f8b95d599de81171', commit_message='Upload dataset', commit_description='', oid='b8c17c666e0c1d0ac1e99dc7f8b95d599de81171', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/aime_train', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/aime_train'), pr_revision=None, pr_num=None)

In [1]:
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset
import os

if not os.path.exists("./AIME"):
    os.makedirs("./AIME")

test_set1 = load_dataset("opencompass/AIME2025", 'AIME2025-I', split="test")
test_set2 = load_dataset("opencompass/AIME2025", 'AIME2025-II', split="test")

with open("./AIME/test.json", "w") as f:
    aime_test = []
    for item in test_set1:
        aime_test.append({
            'Question': item['question'],
            'answer': item['answer'],
        })
    for item in test_set2:
        aime_test.append({
            'Question': item['question'],
            'answer': item['answer'],
        })

    print(len(aime_test))
    json.dump(aime_test, f, ensure_ascii=False, indent=4)


dataset = Dataset.from_list(aime_test)
dataset.push_to_hub('talzoomanzoo/aime_test', private=False)

30


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/aime_test/commit/c70089cb8b81a4411783e9e54bc2831686e92773', commit_message='Upload dataset', commit_description='', oid='c70089cb8b81a4411783e9e54bc2831686e92773', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/aime_test', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/aime_test'), pr_revision=None, pr_num=None)

In [ ]:
import csv
import json
from tqdm import tqdm
from datasets import load_dataset
import os

if not os.path.exists("./simplescaling"):
    os.makedirs("./simplescaling")

train_set = load_dataset("simplescaling/s1K_tokenized", split="train")

with open("./simplescaling/train.json", "w") as f:
    simplescaling_train = []
    for item in train_set:
        # Create new dictionary with only the desired fields
        filtered_item = {
            'Question': item['question'],
            'solution': item['solution'],
            'source_type': item['source_type']
        }
        simplescaling_train.append(filtered_item)
    json.dump(simplescaling_train, f, ensure_ascii=False, indent=4)

In [2]:
"""Data preprocess for MedBullets
Data Link: https://huggingface.co/datasets/LangAGI-Lab/MedBullets
"""
import json
from datasets import load_dataset

train_set = load_dataset("LangAGI-Lab/MedBullets", split="train")
test_set = load_dataset("LangAGI-Lab/MedBullets", split="test")

with open ("./medical/medbullets_train.json", "w") as f:
    medbullets_train = []
    for item in train_set:
        item['Question'] = item.pop('question', None)
        medbullets_train.append(item)
    json.dump(medbullets_train, f, ensure_ascii=False, indent=4)

with open ("./medical/medbullets_test.json", "w") as f:
    medbullets_test = []
    for item in test_set:
        item['Question'] = item.pop('question', None)
        medbullets_test.append(item)
    json.dump(medbullets_test, f, ensure_ascii=False, indent=4)

KeyboardInterrupt: 

In [2]:
"""Data preprocess for JAMA
Data Link: https://huggingface.co/datasets/LangAGI-Lab/jama_full
"""
import json
from datasets import load_dataset

train_set = load_dataset("LangAGI-Lab/jama_full", split="train")
test_set = load_dataset("LangAGI-Lab/jama_full", split="test")

with open ("./medical/jama_full_train.json", "w") as f:
    jama_full_train = []
    for item in train_set:
        item['Question'] = item.pop('question', None)
        jama_full_train.append(item)
    json.dump(jama_full_train, f, ensure_ascii=False, indent=4)

with open ("./medical/jama_full_test.json", "w") as f:
    jama_full_test = []
    for item in test_set:
        item['Question'] = item.pop('question', None)
        jama_full_test.append(item)
    json.dump(jama_full_test, f, ensure_ascii=False, indent=4)

In [ ]:
"""Data preprocess for GPQA
Data Link: https://huggingface.co/datasets/Idavidrein/gpqa
"""
import csv
import json
import random
from tqdm import tqdm

# Paths to data
split = 'diamond'  # diamond, main, extended
data_path = f'./GPQA/original_data/gpqa_{split}.csv'
output_path = f'./GPQA/{split}.json'

# Define the keys we want to keep
keys_to_keep = [
    'id',
    'Question',
    'Subdomain',
    'High-level domain',
    'Correct Answer',
    'Incorrect Answer 1',
    'Incorrect Answer 2',
    'Incorrect Answer 3'
]

filtered_data = []
with open(data_path, mode='r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for idx, row in enumerate(tqdm(csv_reader), 0):
        # Add id field
        row['id'] = idx
        # Create new dictionary with only desired keys
        filtered_row = {key: row[key] for key in keys_to_keep}

        # Extract answers and shuffle them
        answers = [
            ('Correct Answer', filtered_row['Correct Answer']),
            ('Incorrect Answer 1', filtered_row['Incorrect Answer 1']),
            ('Incorrect Answer 2', filtered_row['Incorrect Answer 2']),
            ('Incorrect Answer 3', filtered_row['Incorrect Answer 3'])
        ]
        random.shuffle(answers)

        # Assign new choices A, B, C, D in order and determine the correct choice
        choices = ['A', 'B', 'C', 'D']
        formatted_answers = []
        correct_choice = None
        for i, (label, answer) in enumerate(answers):
            choice = choices[i]
            formatted_answers.append((choice, answer))
            if label == 'Correct Answer':
                correct_choice = choice

        # Update the Question field
        formatted_choices = "\n".join([f"({choice}) {answer}" for choice, answer in formatted_answers])
        filtered_row['Question'] = f"{filtered_row['Question']} Choices:\n{formatted_choices}\n"

        # Add the Correct Choice field
        filtered_row['Correct Choice'] = correct_choice

        # Append the updated row to filtered_data
        filtered_data.append(filtered_row)

# Write the updated data to JSON
with open(output_path, mode='w', encoding='utf-8') as json_file:
    json.dump(filtered_data, json_file, indent=4, ensure_ascii=False)


In [ ]:
"""Data preprocess for MATH500
Data Link: https://huggingface.co/datasets/HuggingFaceH4/MATH-500
"""
import csv
import json
from tqdm import tqdm
import os
from datasets import load_dataset, Dataset

if not os.path.exists("./MATH500"):
    os.makedirs("./MATH500")

test_set = load_dataset("HuggingFaceH4/MATH-500", split="test")

with open("./MATH500/test.json", 'w') as file:
    math500_test = []
    for id, line in enumerate(tqdm(test_set)):
        math500_test.append({
            'id': id, 
            'Question': line['problem'],
            'solution': line['solution'],
            'answer': line['answer'],
            'subject': line['subject'],
            'level': line['level'],
            'unique_id': line['unique_id'],
        })

# Write the updated data to JSON
with open("./MATH500/test.json", mode='w', encoding='utf-8') as json_file:
    json.dump(math500_test, json_file, indent=4, ensure_ascii=False)


ValueError: Unknown split "train". Should be one of ['test'].

In [15]:
"""Data preprocess for AMC
"""
import csv
import json
from tqdm import tqdm

if not os.path.exists("./AMC"):
    os.makedirs("./AMC")

test_set = load_dataset("AI-MO/aimo-validation-amc", split="train")

data_list = []
with open("./AMC/test.json", 'w') as file:
    amc_test = []
    for id, line in enumerate(tqdm(test_set)):
        amc_test.append({
            'id': id, 
            'Question': line['problem'],
            'answer': str(int(line['answer'])),
            'url': line['url'],
        })

# Write the updated data to JSON
with open("./AMC/test.json", mode='w', encoding='utf-8') as json_file:
    json.dump(amc_test, json_file, indent=4, ensure_ascii=False)
print(len(amc_test))


100%|██████████| 83/83 [00:00<00:00, 43015.84it/s]

83


In [23]:
!pip install -U datasets

  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
Using cached datasets-4.1.1-py3-none-any.whl (503 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 2.21.0
    Uninstalling datasets-2.21.0:
      Successfully uninstalled datasets-2.21.0


In [12]:
#!/usr/bin/env python3
"""
Preprocess LiveCodeBench (code_generation_lite)
- Loads test*.jsonl from ./LiveCodeBench if present; otherwise downloads from the HF Hub
- Pins to a specific repo snapshot via commit hash (REVISION), bypassing loader scripts
- Keeps ALL rows (no date filtering, no dedup)
- Serializes `contest_date` safely (handles datetime objects)
- Writes ./LiveCodeBench/test.json
"""

import json
import os
from datetime import datetime, timezone
from typing import Iterable, List, Optional

from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import HfHubHTTPError, EntryNotFoundError
from tqdm import tqdm

# --------- CONFIG ----------
REPO_ID = "livecodebench/code_generation_lite"

# Pin exactly to the commit that corresponds to "feat: release_v5" (Jan 14, 2025).
# Source: https://huggingface.co/datasets/livecodebench/code_generation_lite/commits/main  (hash: 0687ab6)
REVISION = "0687ab6"

LOCAL_DIR = "./LiveCodeBench"             # Set "" to force Hub download
OUTPUT_PATH = "./LiveCodeBench/test.json"

# We list all potential files, then we'll skip any that don't exist at the pinned revision.
CANDIDATE_FILES = [f"test{i}.jsonl" for i in ["", "2", "3", "4", "5", "6"]]
# ---------------------------


# ------------ small helpers ------------

def local_paths(local_dir: str, files: List[str]) -> List[str]:
    """Return existing local file paths under `local_dir`."""
    return [os.path.join(local_dir, f) for f in files if os.path.exists(os.path.join(local_dir, f))]

def hub_paths(files: List[str]) -> List[str]:
    """Download from HF Hub (dataset repo, pinned revision) and return local cache paths.
       Gracefully skip files missing at this revision.
    """
    paths = []
    for f in files:
        try:
            p = hf_hub_download(
                repo_id=REPO_ID,
                filename=f,
                repo_type="dataset",
                revision=REVISION,   # commit hash for "release_v5"
            )
            paths.append(p)
        except (HfHubHTTPError, EntryNotFoundError):
            # File doesn't exist at this snapshot (e.g., test6.jsonl in release_v5) -> skip
            print(f"[warn] Skipping missing file at revision {REVISION}: {f}")
            continue
    if not paths:
        raise RuntimeError(
            f"No files found at revision {REVISION}. "
            "Double-check the commit hash or your network/auth."
        )
    return paths

def load_lcb_as_dataset(local_dir: Optional[str]) -> Dataset:
    """
    Prefer local JSONLs; otherwise fetch specific files from the pinned revision.
    Always return a single Dataset (not a DatasetDict).
    """
    lp = local_paths(local_dir, CANDIDATE_FILES) if local_dir else []
    data_files = lp if lp else hub_paths(CANDIDATE_FILES)
    ds = load_dataset("json", data_files=data_files, split="train")
    return ds

def serialize_contest_date(v) -> str:
    """
    Make sure contest_date is JSON-serializable.
    - datetime -> ISO string (UTC-naive if tz-aware)
    - others   -> str(v)
    """
    if isinstance(v, datetime):
        if v.tzinfo is not None:
            v = v.astimezone(timezone.utc).replace(tzinfo=None)
        return v.isoformat()
    return str(v)


# ------------ write ------------

def write_json(records: Iterable[dict], out_path: str) -> None:
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(list(records), f, ensure_ascii=False, indent=4)


# ------------ main ------------

if __name__ == "__main__":
    print("Loading dataset...")
    ds = load_lcb_as_dataset(LOCAL_DIR)

    print("Collecting rows (no filtering, no dedup)...")
    rows = []
    total = 0
    with tqdm(total=len(ds), desc="Saving all") as pbar:
        for ex in ds:
            total += 1
            rows.append(
                {
                    "id": len(rows),
                    "Question": ex.get("question_content", ""),   # keep even if empty
                    "question_title": ex.get("question_title", ""),
                    "contest_date": serialize_contest_date(ex.get("contest_date", "")),
                    "difficulty": ex.get("difficulty", ""),
                    "public_test_cases": ex.get("public_test_cases", []),
                }
            )
            pbar.update(1)

    print(f"\nTotal rows collected: {len(rows)} (from {total})")

    print("Writing output...")
    write_json(rows, OUTPUT_PATH)
    print(f"Wrote {len(rows)} examples to {OUTPUT_PATH}")


Loading dataset...


[warn] Skipping missing file at revision 0687ab6: test6.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Saving all: 100%|██████████| 880/880 [00:03<00:00, 235.83it/s]


Total rows collected: 880 (from 880)
Writing output...
Wrote 880 examples to ./LiveCodeBench/test.json


In [ ]:
"""Data preprocess for FlashRAG ODQA datasets
Data Link: https://huggingface.co/datasets/RUC-NLPIR/FlashRAG_datasets
"""
import csv
import json
from tqdm import tqdm

dataset_name = 'bamboogle'
split = 'test'
data_num = 500

test_path = f'./FlashRAG_datasets/{dataset_name}/{split}.jsonl'
output_path = f'./QA_Datasets/{dataset_name}.json'

data_list = []
with open(test_path, 'r') as file:
    for id, line in enumerate(tqdm(file.readlines())):
        line = json.loads(line)
        data_list.append({
            'id': id, 
            'Question': line['question'],
            'answer': line["golden_answers"],
        })
        if len(data_list) >= data_num:
            break

# Write the updated data to JSON
with open(output_path, mode='w', encoding='utf-8') as json_file:
    json.dump(data_list, json_file, indent=4, ensure_ascii=False)
